# A PyMC model, sampled directly in the browser kernel

Run all cells in the Xeus environment specified in `.nblink/environment.yml`. Model construction, Numba compilation, Rust sampling and xarray results stay in this one kernel. No iframe or second Python runtime.


In [ ]:
import os, sys
from pathlib import Path
os.environ["PYTENSOR_FLAGS"] = "cxx=,blas__ldflags=,numba__cache=False"
sys.path.insert(0, str(Path.cwd().parent))
import pymc as pm
from notebook_sampler import sample


In [ ]:
with pm.Model() as model:
    mu = pm.Normal("mu", 0, 1)
    sigma = pm.HalfNormal("sigma", 1)
    pm.Normal("observed", mu, sigma, observed=[0.2, -0.1, 0.4, 0.3])

idata = await sample(model, chains=2, tune=250, draws=250)
idata


In [ ]:
idata.posterior["mu"].mean().item(), idata.attrs


## Marketing mix model

The same API accepts the underlying PyMC model built by PyMC-Marketing.


In [ ]:
import pandas as pd
from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation
data = pd.read_csv("mmm/mmm_example.csv", parse_dates=["date_week"])
mmm = MMM(date_column="date_week", channel_columns=["x1", "x2"],
          control_columns=["event_1", "event_2", "t"],
          adstock=GeometricAdstock(l_max=8), saturation=LogisticSaturation(),
          yearly_seasonality=2)
mmm.build_model(data.drop(columns="y"), data.y)
idata = await sample(mmm._get_sampling_model(), chains=2, tune=750, draws=500,
    var_names=["adstock_alpha", "saturation_lam", "saturation_beta",
               "gamma_control", "y_sigma", "intercept_contribution", "gamma_fourier"])
idata


In [ ]:
idata.posterior["adstock_alpha"].mean(("chain", "draw")), idata.attrs


## Explore the posterior interactively

These ArviZ plots use the samples we just generated. Hover to inspect values, drag to zoom, and double-click to reset the axes. The plots are interactive HTML, so no widget server is needed.


In [ ]:
import arviz_plots as azp
from IPython.display import HTML, display


def show_interactive(plot):
    # Inline Plotly's JavaScript so the plots need no CDN or extra renderer extension.
    figure = plot.viz["figure"].item()
    display(HTML(figure.to_html(
        full_html=False, include_plotlyjs=True,
        config={"responsive": True, "displaylogo": False},
    )))


### How much carryover does each channel have?

Compare `adstock_alpha` across channels: larger values mean advertising effects persist longer. `saturation_beta` controls the scale of each channel's modeled contribution.


In [ ]:
posterior_plot = azp.plot_dist(
    idata, var_names=["adstock_alpha", "saturation_beta"], backend="plotly",
)
show_interactive(posterior_plot)


### Follow both sampling chains

Each color is a separate chain. Zoom into the trace to inspect how the carryover estimates move across posterior draws.


In [ ]:
trace_plot = azp.plot_trace(
    idata, var_names=["adstock_alpha"], backend="plotly", aes={"color": ["chain"]},
)
show_interactive(trace_plot)


### Explore parameter trade-offs

Do carryover and contribution scale move together? This pair plot shows their joint posterior for channel `x1`. Change the channel to `x2` and rerun this cell to compare.


In [ ]:
pair_plot = azp.plot_pair(
    idata, var_names=["adstock_alpha", "saturation_beta"],
    coords={"channel": ["x1"]}, backend="plotly",
)
show_interactive(pair_plot)
